<a href="https://colab.research.google.com/github/Kaweri05/Music_Gener_AI/blob/main/Music_Gener_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install music21 pretty_midi tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 32.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.3 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=b13f8675eb05ec38bba39ffac97ebf8e9efa0ef9d5702263a4da72bcbf49f023
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [2]:
from google.colab import files

uploaded = files.upload()

Saving Rothchild Symphony Rmw12 2mov.mid to Rothchild Symphony Rmw12 2mov (1).mid


In [3]:
import glob
from music21 import converter
from music21 import note
from music21 import chord

notes = []

midi_files = glob.glob("*.mid")

print("Files Found:", midi_files)

Files Found: ['Rothchild Symphony Rmw12 2mov.mid', 'Rothchild Symphony Rmw12 2mov (1).mid']


In [4]:
for file in midi_files:

    try:

        midi = converter.parse(file)

        for element in midi.flatten().notes:

            if isinstance(element, note.Note):

                notes.append(
                    str(element.pitch)
                )

            elif isinstance(element, chord.Chord):

                notes.append(
                    ".".join(
                        str(n)
                        for n in element.normalOrder
                    )
                )

    except Exception as e:

        print("Error:", file)

print("Total Notes:", len(notes))

Total Notes: 3236


In [5]:
import pickle

with open(
    "notes.pkl",
    "wb"
) as f:

    pickle.dump(
        notes,
        f
    )

print("Notes Saved")

Notes Saved


In [6]:
import numpy as np

sequence_length = 50

pitchnames = sorted(
    set(notes)
)

note_to_int = dict(
    (
        note,
        number
    )
    for number, note
    in enumerate(pitchnames)
)

network_input = []

network_output = []

for i in range(
        len(notes)
        - sequence_length
):

    sequence_in = notes[
        i:i+sequence_length
    ]

    sequence_out = notes[
        i+sequence_length
    ]

    network_input.append(
        [
            note_to_int[n]
            for n in sequence_in
        ]
    )

    network_output.append(
        note_to_int[
            sequence_out
        ]
    )

print(
    "Patterns:",
    len(network_input)
)

Patterns: 3186


In [7]:
from tensorflow.keras.utils import to_categorical

network_input = np.reshape(
    network_input,
    (
        len(network_input),
        sequence_length,
        1
    )
)

network_input = (
    network_input
    / float(
        len(pitchnames)
    )
)

network_output = to_categorical(
    network_output
)

print(
    network_input.shape
)

(3186, 50, 1)


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

model = Sequential()

model.add(

    LSTM(
        256,
        input_shape=(
            network_input.shape[1],
            network_input.shape[2]
        )
    )

)

model.add(
    Dropout(0.2)
)

model.add(

    Dense(
        network_output.shape[1],
        activation="softmax"
    )

)

model.compile(

    loss="categorical_crossentropy",
    optimizer="adam"

)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 256)            │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 52)             │        13,364 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 277,556 (1.06 MB)

 Trainable params: 277,556 (1.06 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(

    network_input,
    network_output,

    epochs=20,

    batch_size=64

)

Epoch 1/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 202ms/step - loss: 3.6773
Epoch 2/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 216ms/step - loss: 3.6054
Epoch 3/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 216ms/step - loss: 3.6003
Epoch 4/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 200ms/step - loss: 3.5946
Epoch 5/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 217ms/step - loss: 3.5866
Epoch 6/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 218ms/step - loss: 3.5814
Epoch 7/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 23s 262ms/step - loss: 3.5832
Epoch 8/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 193ms/step - loss: 3.5711
Epoch 9/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 207ms/step - loss: 3.5567
Epoch 10/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 217ms/step - loss: 3.5000
Epoch 11/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 13s 258ms/step - loss: 3.4631
Epoch 12/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 13s 260ms/step - loss: 3.4285
Epoch 13/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 232ms/step - loss: 3.3545
Epoch 14/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 215ms/step - loss: 3.3925
Epoch 15/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 

In [10]:
model.save(
    "music_model.keras"
)

print("Model Saved")

Model Saved


In [11]:
from google.colab import files

files.download(
    "music_model.keras"
)

files.download(
    "notes.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import random

start = random.randint(
    0,
    len(network_input)-1
)

pattern = network_input[
    start
]

prediction_output = []

int_to_note = dict(

    (
        number,
        note
    )

    for number,
    note

    in enumerate(
        pitchnames
    )

)